In [1]:
import relbench
from relbench.datasets import get_dataset_names
get_dataset_names()

/home/xuewenqi/relbench/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['rel-amazon',
 'rel-avito',
 'rel-event',
 'rel-f1',
 'rel-hm',
 'rel-stack',
 'rel-mimic',
 'rel-trial',
 'rel-arxiv',
 'rel-salt',
 'rel-ratebeer',
 'dbinfer-avs',
 'dbinfer-diginetica',
 'dbinfer-retailrocket',
 'dbinfer-seznam',
 'dbinfer-amazon',
 'dbinfer-stackexchange',
 'dbinfer-outbrain-small',
 'tgbl-wiki',
 'tgbl-wiki-v2',
 'tgbl-review',
 'tgbl-review-v2',
 'tgbl-coin',
 'tgbl-comment',
 'tgbl-flight',
 'thgl-software',
 'thgl-forum',
 'thgl-github',
 'thgl-myket',
 'tgbn-trade',
 'tgbn-genre',
 'tgbn-reddit',
 'tgbn-token']

In [2]:
from relbench.datasets import get_dataset

dataset = get_dataset(name="rel-f1", download=True)
print(type(dataset))
print(vars(dataset))

<class 'relbench.datasets.f1.F1Dataset'>
{'cache_dir': '/home/xuewenqi/.cache/relbench/rel-f1', 'target_col': None, 'entity_table': None, 'remove_columns': []}


In [3]:
dataset.val_timestamp, dataset.test_timestamp

(Timestamp('2005-01-01 00:00:00'), Timestamp('2010-01-01 00:00:00'))

In [4]:
db = dataset.get_db()
db.min_timestamp, db.max_timestamp

Loading Database object from /home/xuewenqi/.cache/relbench/rel-f1/db...
Done in 0.03 seconds.


(Timestamp('1950-05-13 00:00:00'), Timestamp('2009-11-01 11:00:00'))

In [5]:
table = db.table_dict.keys()
table

dict_keys(['standings', 'constructors', 'circuits', 'races', 'drivers', 'constructor_results', 'results', 'qualifying', 'constructor_standings'])

In [6]:
table = db.table_dict["standings"]
table

Table(df=
       driverStandingsId  raceId  driverId  points  position  wins  \
0                      0       0       789     0.0        20     0   
1                      1       0       640     0.0        18     0   
2                      2       0       589     0.0        19     0   
3                      3       0       669     0.0        15     0   
4                      4       0       661     0.0        22     0   
...                  ...     ...       ...     ...       ...   ...   
28110              28110     819         7    48.0         6     1   
28111              28111     819        68     0.0        25     0   
28112              28112     819        11     0.0        21     0   
28113              28113     819         6     2.0        19     0   
28114              28114     819        12    22.0        11     0   

                     date  
0     1950-05-13 00:00:00  
1     1950-05-13 00:00:00  
2     1950-05-13 00:00:00  
3     1950-05-13 00:00:00  
4     195

In [7]:
table.pkey_col, table.fkey_col_to_pkey_table

('driverStandingsId', {'raceId': 'races', 'driverId': 'drivers'})

In [8]:
from relbench.tasks import get_task_names, get_task
get_task_names("rel-f1")

['driver-position',
 'driver-dnf',
 'driver-top3',
 'driver-circuit-compete',
 'results-position',
 'qualifying-position']

In [9]:
task = get_task("rel-f1", "driver-top3", download=False)
task.get_table("train")

Table(df=
           date  driverId  qualifying
0    1994-02-28        43           0
1    1994-02-28        56           0
2    1994-03-30       101           1
3    1994-03-30       103           0
4    1994-04-29        70           1
...         ...       ...         ...
1348 2004-04-06        12           0
1349 2004-05-06        30           1
1350 2004-05-06        14           1
1351 2004-05-06        17           1
1352 2004-05-06        44           0

[1353 rows x 3 columns],
  fkey_col_to_pkey_table={'driverId': 'drivers'},
  pkey_col=None,
  time_col=date)

In [10]:
import torch 
import os
import relbench
import numpy as np
from torch.nn import BCEWithLogitsLoss, L1Loss
from relbench.datasets import get_dataset
from relbench.tasks import get_task

In [11]:
dataset = get_dataset("rel-f1", download=False)
task = get_task("rel-f1", "driver-position", download=True)

In [12]:
train_table = task.get_table("train")
val_table = task.get_table("val")
test_table = task.get_table("test")
out_channals = 1
loss_fn = L1Loss()
tune_metric = "mae"
higher_is_better = False

In [13]:
train_table

Table(df=
           date  driverId  position
0    2004-07-05        10     10.75
1    2004-07-05        47     12.00
2    2004-03-07         7     15.00
3    2004-01-07        10      9.00
4    2003-09-09        52     13.00
...         ...       ...       ...
7448 1995-08-22        96     15.75
7449 1975-06-08       228      8.00
7450 1965-05-31       418     16.00
7451 1961-08-20       467     37.00
7452 1954-05-29       677     30.00

[7453 rows x 3 columns],
  fkey_col_to_pkey_table={'driverId': 'drivers'},
  pkey_col=None,
  time_col=date)

In [14]:
import os 
import math
import numpy as np
from tqdm import tqdm

import torch 
import torch_geometric
import torch_frame


from torch_geometric.seed import seed_everything
seed_everything(42) 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
root_dir= "./data"


cuda


In [15]:
from relbench.modeling.utils import get_stype_proposal
db = dataset.get_db()
col_to_stype_dict = get_stype_proposal(db)
col_to_stype_dict

Loading Database object from /home/xuewenqi/.cache/relbench/rel-f1/db...
Done in 0.03 seconds.


{'standings': {'driverStandingsId': <stype.numerical: 'numerical'>,
  'raceId': <stype.numerical: 'numerical'>,
  'driverId': <stype.numerical: 'numerical'>,
  'points': <stype.numerical: 'numerical'>,
  'position': <stype.numerical: 'numerical'>,
  'wins': <stype.numerical: 'numerical'>,
  'date': <stype.timestamp: 'timestamp'>},
 'constructors': {'constructorId': <stype.numerical: 'numerical'>,
  'constructorRef': <stype.text_embedded: 'text_embedded'>,
  'name': <stype.text_embedded: 'text_embedded'>,
  'nationality': <stype.text_embedded: 'text_embedded'>},
 'circuits': {'circuitId': <stype.numerical: 'numerical'>,
  'circuitRef': <stype.text_embedded: 'text_embedded'>,
  'name': <stype.text_embedded: 'text_embedded'>,
  'location': <stype.text_embedded: 'text_embedded'>,
  'country': <stype.text_embedded: 'text_embedded'>,
  'lat': <stype.numerical: 'numerical'>,
  'lng': <stype.numerical: 'numerical'>,
  'alt': <stype.numerical: 'numerical'>},
 'races': {'raceId': <stype.numerica

In [16]:
from typing import List, Optional
from sentence_transformers import SentenceTransformer
from torch import Tensor
class GloveTextEmbedding: 
    def __init__(self, device: Optional[torch.device] = None): 
        self.model = SentenceTransformer(
            "sentence-transformers/average_word_embeddings_glove.6B.300d",
            device=device, 
        )
    def __call__(self, sentences: List[str]) -> Tensor: 
        return torch.from_numpyz(self.model.encode(sentences))

In [19]:
from torch_frame.config.text_embedder import TextEmbedderConfig
from relbench.modeling.graph import make_pkey_fkey_graph
text_embedding_cfg = TextEmbedderConfig(
    text_embedder=GloveTextEmbedding(device=device), batch_size=256
)
data, col_stats_dict = make_pkey_fkey_graph(
    db, 
    col_to_stype_dict=col_to_stype_dict, 
    text_embedder_cfg=text_embedding_cfg, 
    cache_dir=os.path.join(
        root_dir, f"rel-f1_materialized_cache"
    ),  # store materialized graph for convenience
)


/home/xuewenqi/relbench/.venv/lib/python3.10/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([scalar])` to allowlist this global.
  self._tensor_frame, self._col_stats = torch_frame.load(
/home/xuewenqi/relbench/.venv/lib/python3.10/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([scalar])` to allowlist this global.
  self._tensor_frame, self._col_stats = torch_frame.load(
/home/xuewenqi/relbench/.venv/lib/python3.10/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([scalar])` t